# Food Recognition CNN: Colab Notebook

This notebook trains a CNN that recognises fruits and vegetables from a photo. It connects a dataset to the code, builds two models, and compares them:

1. **Baseline CNN** built from scratch.
2. **Transfer-learning model** (MobileNetV2 pre-trained on ImageNet).

**Before you start**
1. In the menu choose **Runtime > Change runtime type > T4 GPU**.
2. Run the cells from top to bottom. Cells marked **[COLAB]** only work inside Google Colab.
3. Choose **one** way to load the data in Section 2 (Kaggle, Google Drive, or your own zip file), or use **Section 2B** to fetch datasets for all 18 food categories.

**Expected folder layout** (class name = folder name):

```
dataset/
  train/        tomato/  onion/  potato/ ...
  validation/   tomato/  onion/  potato/ ...   (optional)
  test/         tomato/  onion/  potato/ ...   (optional)
```

A single folder of class folders also works. The notebook then makes its own 70/15/15 split.

## 1. Setup

In [ ]:
# [COLAB] Install anything Colab might be missing (usually nothing is needed)
!pip install -q scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\moham\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


: 

In [ ]:
import os, json, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPU found:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ---- Settings you can change ----
SEED = 42
IMG_SIZE = (160, 160)       # image size fed to the CNN
BATCH_SIZE = 32
EPOCHS_BASELINE = 20        # baseline CNN
EPOCHS_TRANSFER = 10        # transfer learning (base frozen)
EPOCHS_FINETUNE = 10        # fine-tuning (top layers unfrozen)
MAX_PER_CLASS = 300         # use at most this many images per class (keeps training fast). None = use all
WEIGHTS = "imagenet"        # pre-trained weights for MobileNetV2

DATA_ROOT = "/content/data"  # where the dataset is stored (changed in Section 2 if you use Drive)

keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

## 2. Connect the dataset

Pick **one** option and run only that option.

| Option | Use when |
|---|---|
| **A. Kaggle** | You want a public dataset quickly |
| **B. Google Drive** | Your dataset (or your own photos) is already in Drive |
| **C. Upload a zip** | Small dataset, quick test |

**Option A needs a Kaggle API key:** on kaggle.com go to *Settings > API > Create New Token*. This downloads a file called `kaggle.json`. You will upload it below.

Two datasets that work well:
- `kritikseth/fruit-and-vegetable-image-recognition` (36 classes, real-world style photos)
- `moltean/fruits` (Fruits-360, clean photos on white background)

If a download fails, open the dataset page on Kaggle and copy the part of the URL after `kaggle.com/datasets/`.

In [ ]:
# [COLAB] OPTION A: download from Kaggle
from google.colab import files
print("Choose your kaggle.json file")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

KAGGLE_DATASET = "kritikseth/fruit-and-vegetable-image-recognition"
!kaggle datasets download -d $KAGGLE_DATASET -p /content/data --unzip -q
DATA_ROOT = "/content/data"

In [ ]:
# [COLAB] OPTION B: use a dataset stored in Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Change this to the folder in your Drive that contains the class folders (or train/test folders)
DATA_ROOT = "/content/drive/MyDrive/food_dataset"

In [ ]:
# [COLAB] OPTION C: upload a zip file from your computer
from google.colab import files
import zipfile
uploaded = files.upload()          # choose your .zip file
for name in uploaded:
    with zipfile.ZipFile(name) as z:
        z.extractall("/content/data")
DATA_ROOT = "/content/data"

## 2B. Fetch datasets for all 18 categories (alternative to Options A to C)

Use this section when you want **all 18 categories** of your label taxonomy. It downloads many datasets, maps their classes onto **your** item names, and builds one folder:

```
/content/data/<item>/image files      (one folder per item, e.g. tomato, paneer, soy_sauce)
/content/class_to_category.csv        (item -> category)
```

**How the data is fetched**

| Method | What it does | Needs |
|---|---|---|
| `kaggle` | Downloads a Kaggle dataset | Your `kaggle.json` (you upload it once) |
| `hf` | Downloads only the needed class folders from Hugging Face | Nothing |
| `git` | Clones a GitHub repository | Nothing |
| `manual` | Reads a zip you downloaded in your browser (Mendeley and similar sites do not allow scripted downloads) | Put `<id>.zip` in your Drive folder `food_datasets_manual` |
| `off` | Pulls product photos from the Open Food Facts API for packaged items | Nothing (be polite: small caps, delays) |

**Be aware**
- **No dataset covers every item.** Items with no image source (for example raw chicken, beef, pork, kale, millet) show as **MISSING** in the report. Fill them with your own phone photos by adding a folder under `/content/data/<item>/`.
- Open Food Facts gives **packaging photos**, so those classes learn the look of a pack or bottle, not the raw food.
- Kaggle slugs marked `verified=False` were not confirmed on the site. If one fails, search its name on Kaggle and fix the `ref`.
- An image classifier can say "milk" or "cooking oil", but not the brand or nutrition facts. For branded products, scan the **barcode** or read the **label** and look the product up (for example in Open Food Facts).
- Check every dataset's licence before you use it (see the Dataset Sources document).

In [ ]:
import re, shutil, subprocess, zipfile, time, csv
import requests
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# ---- Settings for the fetch ----
OUT_DIR = "/content/data"                                  # unified output (one folder per item)
RAW_DIR = "/content/raw"                                   # raw downloads
MANUAL_DIR = "/content/drive/MyDrive/food_datasets_manual" # put manual zips here (named <id>.zip)
MAX_PER_SOURCE_ITEM = 300     # max images taken from one source for one item
OFF_MAX_PER_ITEM = 150        # max Open Food Facts photos per item
MIN_IMAGES_PER_CLASS = 30     # items with fewer images are moved aside (too few to train on)
OFF_CONTACT = "your-email@example.com"   # Open Food Facts asks for a contact in the User-Agent. Put yours.
FETCH_ENABLED = {"kaggle": True, "hf": True, "git": True, "manual": True, "off": True, "food101": False}
# food101 streams a ~5 GB dataset (slow). Turn it on only if you need soup, salad and pasta dish photos.

# ---- Your target taxonomy: category -> items ----
TAXONOMY = {
    "Fruits": ["apple", "banana", "mango", "orange", "grape", "watermelon", "pineapple", "papaya", "guava", "avocado", "lemon", "lime", "strawberry", "pomegranate"],
    "Vegetables": ["tomato", "potato", "onion", "carrot", "cucumber", "cabbage", "cauliflower", "eggplant", "capsicum", "beetroot", "pumpkin", "zucchini"],
    "Leafy greens and fresh herbs": ["spinach", "lettuce", "kale", "coriander leaves", "mint", "curry leaves", "fenugreek leaves"],
    "Grains and cereals": ["rice", "wheat", "oats", "corn", "barley", "millet", "ragi", "quinoa"],
    "Pulses, beans and lentils": ["chickpeas", "green gram", "black gram", "kidney beans", "lentils", "peas", "pigeon peas", "soybeans"],
    "Nuts and seeds": ["almonds", "cashews", "peanuts", "walnuts", "pistachios", "sesame", "chia", "flaxseed", "sunflower seeds"],
    "Dairy products": ["milk", "curd", "yoghurt", "cheese", "paneer", "butter", "ghee", "cream"],
    "Milk alternatives": ["soy milk", "almond milk", "oat milk", "coconut milk"],
    "Eggs and animal proteins": ["eggs", "chicken", "beef", "mutton", "pork", "tofu", "tempeh"],
    "Fish and seafood": ["sardines", "tuna", "salmon", "mackerel", "anchovies", "prawns", "crab", "mussels"],
    "Oils and fats": ["coconut oil", "sunflower oil", "olive oil", "groundnut oil", "mustard oil", "sesame oil"],
    "Spices and seasonings": ["turmeric", "chilli powder", "black pepper", "cumin", "coriander seeds", "cinnamon", "cardamom", "cloves", "garam masala"],
    "Sweeteners and baking ingredients": ["sugar", "jaggery", "honey", "flour", "baking powder", "cocoa powder", "yeast"],
    "Sauces and condiments": ["ketchup", "mayonnaise", "mustard", "soy sauce", "vinegar", "chutney", "pickles"],
    "Packaged staple foods": ["rice packets", "flour bags", "packaged oats", "pasta", "noodles", "bread", "breakfast cereals"],
    "Packaged snacks and convenience foods": ["biscuits", "chips", "crackers", "instant noodles", "frozen foods", "ready-to-eat meals"],
    "Beverages": ["water", "fruit juice", "soft drinks", "tea", "coffee", "packaged milk drinks"],
    "Prepared meals and dishes": ["biryani", "dosa", "idli", "chapati", "dal", "curry", "sambar", "salad", "soup", "pasta dishes"],
}
# (Ghee also belongs under Oils and fats. It is kept once, under Dairy products.)

# Extra names that should map to an item (matched against the dataset's own folder names)
EXTRA_ALIASES = {
    "capsicum": ["bell pepper", "paprika", "pepper"], "eggplant": ["aubergine", "brinjal"], "zucchini": ["courgette"],
    "grape": ["grapes"], "tomato": ["tomatoes"], "potato": ["potatoes"],
    "coriander leaves": ["coriander", "cilantro", "coriander leaf"], "curry leaves": ["curry leaf", "curry tree", "curry", "murraya", "bergera koenigii"],
    "fenugreek leaves": ["fenugreek", "methi", "trigonella"], "mint": ["mentha", "pudina"], "lettuce": ["salad leaf"],
    "corn": ["sweetcorn", "maize"], "rice": ["basmati", "jasmine", "arborio"],
    "chickpeas": ["chickpea", "chana", "kabuli", "chole"], "green gram": ["mung", "moong", "mugh", "mug", "whole green gram"],
    "black gram": ["urad", "mashkalai", "mash"], "kidney beans": ["rajma", "kidney bean", "red beans"], "lentils": ["lentil", "masoor", "masur"],
    "pigeon peas": ["pigeon pea", "orhor", "arhar", "toor", "tur"], "peas": ["pea", "motor", "green peas"],
    "soybeans": ["soy beans", "soya beans", "soya bean", "soyabean", "soy bean"],
    "almonds": ["almond"], "cashews": ["cashew", "cashew nut"], "peanuts": ["peanut", "groundnut"], "walnuts": ["walnut"], "pistachios": ["pistachio"],
    "yoghurt": ["yogurt", "yoghurts", "yogurts"], "curd": ["dahi", "curds"], "eggs": ["egg"],
    "soy milk": ["soya milk", "soy drink"], "oat milk": ["oat drink", "oatly"], "almond milk": ["almond drink"], "coconut milk": ["coconut drink"],
    "prawns": ["shrimp", "shrimps", "prawn"], "mackerel": ["mackerels"], "sardines": ["sardine"], "mussels": ["mussel"],
    "chilli powder": ["chili powder", "red chilli powder"], "black pepper": ["peppercorn"], "cumin": ["jeera", "cumin seeds"],
    "coriander seeds": ["coriander", "dhania", "coriander seed"], "cloves": ["clove"], "turmeric": ["haldi"],
    "fruit juice": ["juice", "juices"], "soft drinks": ["soda", "cola", "soft drink"], "water": ["mineral water"],
    "chapati": ["roti", "chappati", "phulka"], "dal": ["daal", "dhal"], "pasta dishes": ["spaghetti", "lasagna", "macaroni", "ravioli"],
}

# Source catalog. kind: kaggle / hf / git / manual.  mode: match (map by name) / explicit (map table) / fixed (everything -> one item)
CATALOG = [
    dict(id="fruits360", kind="kaggle", ref="moltean/fruits", verified=True, mode="match",
         cats=["Fruits", "Vegetables"], link="https://www.kaggle.com/datasets/moltean/fruits"),
    dict(id="fv36", kind="kaggle", ref="kritikseth/fruit-and-vegetable-image-recognition", verified=False, mode="match",
         cats=["Fruits", "Vegetables", "Leafy greens and fresh herbs", "Pulses, beans and lentils", "Grains and cereals", "Spices and seasonings"],
         link="https://hyper.ai/en/datasets/32609"),
    dict(id="nuts11", kind="kaggle", ref="ruopengan/11-common-nut-types-for-image-classification", verified=True, mode="match",
         cats=["Nuts and seeds"], link="https://www.kaggle.com/datasets/ruopengan/11-common-nut-types-for-image-classification"),
    dict(id="rice5", kind="kaggle", ref="muratkokludataset/rice-image-dataset", verified=False, mode="fixed", item="rice",
         link="https://www.muratkoklu.com/datasets/"),
    dict(id="fish9", kind="kaggle", ref="crowww/a-large-scale-fish-dataset", verified=False, mode="explicit",
         map={"shrimp": "prawns", "horse_mackerel": "mackerel"}, exclude=r"(^|_)gt$",
         link="https://ml.paperswithcode.com/dataset/a-large-scale-fish-dataset"),
    dict(id="indianfood_kaggle", kind="kaggle", ref="l33tc0d3r/indian-food-classification", verified=True, mode="explicit",
         map={"chapati": "chapati", "dal_makhani": "dal", "idli": "idli", "masala_dosa": "dosa", "kadai_paneer": "curry"},
         link="https://www.kaggle.com/datasets/l33tc0d3r/indian-food-classification"),
    dict(id="indianfood_hf", kind="hf", ref="SKSKCulinaryAI/Indian_Food_Images_Dataset", verified=True, mode="explicit",
         patterns=["train/Biryani/*", "train/Chappati/*", "train/Dosa/*", "train/Idli/*", "train/Dal makhani/*", "train/Toor dal/*", "train/Butter chicken/*", "train/Kadai_paneer/*"],
         map={"biryani": "biryani", "chappati": "chapati", "dosa": "dosa", "idli": "idli", "dal_makhani": "dal", "toor_dal": "dal", "butter_chicken": "curry", "kadai_paneer": "curry"},
         link="https://huggingface.co/datasets/SKSKCulinaryAI/Indian_Food_Images_Dataset"),
    dict(id="grocery", kind="git", ref="https://github.com/marcusklasson/GroceryStoreDataset.git", verified=False, mode="match",
         cats=["Fruits", "Vegetables", "Dairy products", "Milk alternatives", "Beverages"],
         link="https://github.com/marcusklasson/GroceryStoreDatasetPaper"),
    # ---- manual: download the zip in your browser, save it as <id>.zip in MANUAL_DIR ----
    dict(id="indian_lentils", kind="manual", mode="match", cats=["Pulses, beans and lentils"], link="https://data.mendeley.com/datasets/r4yhfnzc5y/1"),
    dict(id="bdlentil", kind="manual", mode="match", cats=["Pulses, beans and lentils"], link="https://data.mendeley.com/datasets/ksxcmtwbhy"),
    dict(id="dryfruit", kind="manual", mode="match", cats=["Nuts and seeds"], link="https://data.mendeley.com/datasets/yfhgn8py5f"),
    dict(id="allernuts", kind="manual", mode="match", cats=["Nuts and seeds"], link="https://data.mendeley.com/datasets/3gtbxvgm5f"),
    dict(id="spice_spectrum", kind="manual", mode="match", cats=["Spices and seasonings"], link="https://doi.org/10.17632/5v7w2hx8n5.2"),
    dict(id="spice_visual", kind="manual", mode="match", cats=["Spices and seasonings"], link="https://data.mendeley.com/datasets/h9pphm522g"),
    dict(id="spice_indian", kind="manual", mode="match", cats=["Spices and seasonings"], link="https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11827071/"),
    dict(id="leaf_medicinal", kind="manual", mode="match", cats=["Leafy greens and fresh herbs"], link="https://data.mendeley.com/datasets/k8m93ft9kk/2"),
    dict(id="leaf_bd", kind="manual", mode="match", cats=["Leafy greens and fresh herbs"], link="https://data.mendeley.com/datasets/fj93rrfv2y/2"),
    dict(id="herb_species", kind="manual", mode="match", cats=["Leafy greens and fresh herbs"], link="https://data.mendeley.com/datasets/fmpf798zcn"),
    dict(id="rice_aruzz", kind="manual", mode="fixed", item="rice", link="https://data.mendeley.com/datasets/3mn9843tz2/3"),
    dict(id="khana", kind="manual", mode="match", cats=["Prepared meals and dishes"], link="https://khana.omkar.xyz"),
]

# Items filled from Open Food Facts (packaged products). Each list = category names to try, in order.
OFF_QUERIES = {
    "milk": ["Milks"], "curd": ["Curd", "Curds", "Dahi"], "yoghurt": ["Yogurts", "Yoghurts"], "cheese": ["Cheeses"], "paneer": ["Paneer"],
    "butter": ["Butters"], "ghee": ["Ghee", "Clarified butters"], "cream": ["Creams"],
    "soy milk": ["Soy milks", "Soy drinks"], "almond milk": ["Almond milks", "Almond drinks"], "oat milk": ["Oat milks", "Oat drinks"], "coconut milk": ["Coconut milks"],
    "eggs": ["Eggs"], "tofu": ["Tofu"], "tempeh": ["Tempeh"],
    "sardines": ["Sardines"], "tuna": ["Tunas", "Tuna"], "salmon": ["Salmons"], "mackerel": ["Mackerels"], "anchovies": ["Anchovies"],
    "prawns": ["Shrimps", "Prawns"], "crab": ["Crabs"], "mussels": ["Mussels"],
    "coconut oil": ["Coconut oils"], "sunflower oil": ["Sunflower oils"], "olive oil": ["Olive oils"], "groundnut oil": ["Groundnut oils", "Peanut oils"],
    "mustard oil": ["Mustard oils"], "sesame oil": ["Sesame oils"],
    "sesame": ["Sesame seeds"], "chia": ["Chia seeds"], "flaxseed": ["Flax seeds", "Flaxseeds"], "sunflower seeds": ["Sunflower seeds"],
    "oats": ["Rolled oats", "Oat flakes"], "barley": ["Barley"], "millet": ["Millets"], "ragi": ["Ragi", "Finger millet"], "quinoa": ["Quinoa"], "wheat": ["Wheat"],
    "chilli powder": ["Chili powders", "Chilli powder"], "garam masala": ["Garam masala"],
    "sugar": ["Sugars"], "jaggery": ["Jaggery"], "honey": ["Honeys"], "flour": ["Wheat flours", "Flours"], "baking powder": ["Baking powders"],
    "cocoa powder": ["Cocoa powders", "Cocoa"], "yeast": ["Yeasts", "Yeast"],
    "ketchup": ["Ketchup", "Tomato ketchup"], "mayonnaise": ["Mayonnaises"], "mustard": ["Mustards"], "soy sauce": ["Soy sauces"],
    "vinegar": ["Vinegars"], "chutney": ["Chutneys"], "pickles": ["Pickles"],
    "rice packets": ["Rices", "Basmati rices"], "flour bags": ["Wheat flours"], "packaged oats": ["Rolled oats"], "pasta": ["Pastas"],
    "noodles": ["Noodles"], "bread": ["Breads"], "breakfast cereals": ["Breakfast cereals"],
    "biscuits": ["Biscuits"], "chips": ["Chips and fries", "Crisps"], "crackers": ["Crackers"], "instant noodles": ["Instant noodles"],
    "frozen foods": ["Frozen foods"], "ready-to-eat meals": ["Meals", "Prepared meals"],
    "water": ["Waters"], "fruit juice": ["Fruit juices"], "soft drinks": ["Sodas", "Carbonated drinks"], "tea": ["Teas"], "coffee": ["Coffees"],
    "packaged milk drinks": ["Flavoured milks", "Milk drinks"],
}
print(sum(len(v) for v in TAXONOMY.values()), "items in", len(TAXONOMY), "categories")

In [ ]:
def norm(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def _sing(tok):
    return tok[:-1] if len(tok) > 3 and tok.endswith("s") and not tok.endswith("ss") else tok

def toks(s):
    return [_sing(t) for t in norm(s).split("_") if t]

def _find_end(name_toks, alias_toks):
    """Index of the last token of the LAST place where alias_toks appears in name_toks, or -1."""
    n = len(alias_toks)
    ends = [i + n - 1 for i in range(len(name_toks) - n + 1) if n > 0 and name_toks[i:i + n] == alias_toks]
    return ends[-1] if ends else -1

ITEM_ALIASES = {}
for _cat, _items in TAXONOMY.items():
    for _it in _items:
        ITEM_ALIASES[_it] = [_it] + EXTRA_ALIASES.get(_it, [])
ITEM_CATEGORY = {}
for _cat, _items in TAXONOMY.items():
    for _it in _items:
        ITEM_CATEGORY.setdefault(_it, _cat)


def match_item(class_name, cats):
    """Map a dataset class name to one of OUR items.
    The match that ends last in the name wins (the main noun comes last: 'Apple Juice' -> juice),
    and a longer alias beats a shorter one ('soy milk' beats 'milk')."""
    name_t = toks(class_name)
    best = None
    for cat in cats:
        for item in TAXONOMY[cat]:
            for alias in ITEM_ALIASES[item]:
                at = toks(alias)
                end = _find_end(name_t, at)
                if end >= 0 and (best is None or (end, len(at)) > best[0]):
                    best = ((end, len(at)), item)
    return best[1] if best else None


def map_class(src, class_name):
    mode = src["mode"]
    if mode == "match":
        return match_item(class_name, src["cats"])
    if mode == "explicit":
        return src["map"].get(norm(class_name))
    if mode == "fixed":
        return src["item"]
    return None


def place(path, item, tag, k):
    """Put one image into OUT_DIR/<item>/ (symlink, or copy if links are not allowed)."""
    folder = os.path.join(OUT_DIR, norm(item))
    os.makedirs(folder, exist_ok=True)
    ext = os.path.splitext(path)[1].lower() or ".jpg"
    dst = os.path.join(folder, f"{tag}__{k:05d}{ext}")
    if not os.path.exists(dst):
        try:
            os.symlink(os.path.abspath(path), dst)
        except OSError:
            shutil.copy(path, dst)


def ingest_folder(src, folder):
    """Find every folder of images under `folder`, map its name to an item, and place the images."""
    per_class = defaultdict(list)
    for root, _, files_ in os.walk(folder):
        imgs = [f for f in files_ if f.lower().endswith(IMG_EXT)]
        if imgs:
            per_class[os.path.basename(root)] += [os.path.join(root, f) for f in imgs]
    added, skipped, k = defaultdict(int), [], defaultdict(int)
    for cname in sorted(per_class):
        if src.get("exclude") and re.search(src["exclude"], norm(cname)):
            continue
        item = map_class(src, cname)
        if not item:
            skipped.append(cname)
            continue
        paths = sorted(per_class[cname])
        random.Random(SEED).shuffle(paths)
        for p in paths:
            if added[item] >= MAX_PER_SOURCE_ITEM:
                break
            place(p, item, src["id"], k[item])
            k[item] += 1
            added[item] += 1
    return dict(added), skipped


# ---------- downloaders ----------
_kaggle_ready = None

def ensure_kaggle_auth():
    global _kaggle_ready
    if _kaggle_ready is not None:
        return _kaggle_ready
    p = os.path.expanduser("~/.kaggle/kaggle.json")
    if os.path.exists(p) or os.environ.get("KAGGLE_KEY"):
        _kaggle_ready = True
        return True
    try:
        from google.colab import files as colab_files
        print("Upload your kaggle.json (Kaggle > Settings > API > Create New Token)")
        up = colab_files.upload()
        os.makedirs(os.path.dirname(p), exist_ok=True)
        with open(p, "wb") as f:
            f.write(next(iter(up.values())))
        os.chmod(p, 0o600)
        _kaggle_ready = True
    except Exception as e:
        print("Kaggle login is not set up:", e)
        _kaggle_ready = False
    return _kaggle_ready


def fetch_kaggle(src):
    dest = os.path.join(RAW_DIR, src["id"])
    if os.path.isdir(dest) and os.listdir(dest):
        return dest
    if not ensure_kaggle_auth():
        raise RuntimeError("no Kaggle login")
    os.makedirs(dest, exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", src["ref"], "-p", dest, "--unzip", "-q"], check=True)
    return dest


def fetch_hf(src):
    from huggingface_hub import snapshot_download
    dest = os.path.join(RAW_DIR, src["id"])
    snapshot_download(repo_id=src["ref"], repo_type="dataset", local_dir=dest, allow_patterns=src.get("patterns"))
    return dest


def fetch_git(src):
    dest = os.path.join(RAW_DIR, src["id"])
    if not os.path.isdir(dest):
        subprocess.run(["git", "clone", "--depth", "1", src["ref"], dest], check=True)
    return dest


def fetch_manual(src):
    """Use <MANUAL_DIR>/<id>.zip (or an extracted folder). Returns None if it is not there."""
    dest = os.path.join(RAW_DIR, src["id"])
    if os.path.isdir(dest) and os.listdir(dest):
        return dest
    zip_path = os.path.join(MANUAL_DIR, src["id"] + ".zip")
    folder_path = os.path.join(MANUAL_DIR, src["id"])
    if os.path.exists(zip_path):
        os.makedirs(dest, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(dest)
        return dest
    if os.path.isdir(folder_path):
        return folder_path
    return None


FETCHERS = {"kaggle": fetch_kaggle, "hf": fetch_hf, "git": fetch_git, "manual": fetch_manual}


# ---------- Open Food Facts (packaged products) ----------
def _ua():
    return {"User-Agent": f"FoodRecognitionStudentProject/1.0 ({OFF_CONTACT})"}

def _get(url, **kw):
    for attempt in range(4):
        try:
            r = requests.get(url, headers=_ua(), timeout=60, **kw)
            if r.status_code == 200:
                return r
        except requests.RequestException:
            pass
        time.sleep(2 * (attempt + 1))      # Open Food Facts sometimes answers 503: wait and retry
    return None

_off_cats = None

def off_categories():
    global _off_cats
    if _off_cats is None:
        r = _get("https://world.openfoodfacts.org/categories.json")
        best = {}
        if r is not None:
            for t in r.json().get("tags", []):
                nm = str(t.get("name", "")).lower()
                if nm and (nm not in best or t.get("products", 0) > best[nm].get("products", 0)):
                    best[nm] = t
        _off_cats = best
    return _off_cats


def resolve_off_tag(labels):
    cats = off_categories()
    for lab in labels:                      # 1) exact name
        t = cats.get(lab.lower())
        if t:
            return t["id"]
    for lab in labels:                      # 2) name that contains the label as whole words (most products wins)
        pat = re.compile(r"\b" + re.escape(lab.lower()) + r"\b")
        hits = [t for nm, t in cats.items() if pat.search(nm)]
        if hits:
            return max(hits, key=lambda t: t.get("products", 0))["id"]
    return None


def fetch_off_item(item, labels, cap):
    tag = resolve_off_tag(labels)
    if not tag:
        return 0, None
    found, page = [], 1
    while len(found) < cap and page <= 5:
        r = _get("https://world.openfoodfacts.org/api/v2/search",
                 params=dict(categories_tags=tag, fields="code,image_front_small_url,image_front_url", page_size=100, page=page))
        prods = r.json().get("products", []) if r is not None else []
        if not prods:
            break
        for p in prods:
            u = p.get("image_front_small_url") or p.get("image_front_url")
            if u and p.get("code"):
                found.append((p["code"], u))
        page += 1
        time.sleep(0.6)
    found = found[:cap]
    folder = os.path.join(OUT_DIR, norm(item))
    os.makedirs(folder, exist_ok=True)

    def download(cu):
        code_, url = cu
        dst = os.path.join(folder, f"off__{code_}.jpg")
        if os.path.exists(dst):
            return 1
        r = _get(url)
        if r is None or not r.headers.get("content-type", "").startswith("image"):
            return 0
        with open(dst, "wb") as f:
            f.write(r.content)
        return 1

    with ThreadPoolExecutor(max_workers=6) as ex:
        n = sum(ex.map(download, found))
    return n, tag


# ---------- optional: Food-101 (soup, salad, pasta dishes) ----------
FOOD101_MAP = {"caesar_salad": "salad", "greek_salad": "salad", "caprese_salad": "salad", "beet_salad": "salad",
               "french_onion_soup": "soup", "miso_soup": "soup", "hot_and_sour_soup": "soup",
               "spaghetti_bolognese": "pasta dishes", "spaghetti_carbonara": "pasta dishes", "lasagna": "pasta dishes", "ravioli": "pasta dishes",
               "chicken_curry": "curry"}

def fetch_food101(cap):
    from datasets import load_dataset
    ds = load_dataset("ethz/food101", split="train", streaming=True)
    names = ds.features["label"].names
    counts = defaultdict(int)
    wanted = set(FOOD101_MAP.values())
    for ex in ds:
        item = FOOD101_MAP.get(names[ex["label"]])
        if not item or counts[item] >= cap:
            continue
        folder = os.path.join(OUT_DIR, norm(item))
        os.makedirs(folder, exist_ok=True)
        ex["image"].convert("RGB").save(os.path.join(folder, f"food101__{counts[item]:05d}.jpg"))
        counts[item] += 1
        if all(counts[i] >= cap for i in wanted):
            break
    return dict(counts)


def run_fetch():
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(RAW_DIR, exist_ok=True)
    problems = []
    for src in CATALOG:
        kind = src["kind"]
        if not FETCH_ENABLED.get(kind, True):
            continue
        try:
            folder = FETCHERS[kind](src)
            if folder is None:
                print(f"[skip]   {src['id']}: zip not found. Download from {src['link']} and save as {src['id']}.zip in {MANUAL_DIR}")
                continue
            added, skipped = ingest_folder(src, folder)
            warn = "" if src.get("verified", True) or kind != "kaggle" else "  (slug not verified)"
            print(f"[ok]     {src['id']}: {sum(added.values())} images into {len(added)} items; {len(skipped)} folders not in your taxonomy{warn}")
        except Exception as e:
            problems.append(src["id"])
            print(f"[FAILED] {src['id']}: {e}  -> open {src['link']}")
    if FETCH_ENABLED.get("off", True):
        for item, labels in OFF_QUERIES.items():
            try:
                n, tag = fetch_off_item(item, labels, OFF_MAX_PER_ITEM)
                print(f"[off]    {item}: {n} photos" if tag else f"[off]    {item}: category not found for {labels}")
            except Exception as e:
                problems.append("off:" + item)
                print(f"[FAILED] off {item}: {e}")
    if FETCH_ENABLED.get("food101", False):
        print("[food101]", fetch_food101(MAX_PER_SOURCE_ITEM))
    return problems


def count_images(item):
    folder = os.path.join(OUT_DIR, norm(item))
    if not os.path.isdir(folder):
        return 0
    return sum(f.lower().endswith(IMG_EXT) for f in os.listdir(folder))


def coverage_report(move_low=True):
    """Print images per item, write class_to_category.csv, and move items with too few images aside."""
    low_dir = OUT_DIR + "_low"
    rows, ready = [], 0
    for cat, items in TAXONOMY.items():
        n_ready = 0
        print(f"\n{cat}")
        for it in items:
            n = count_images(it)
            status = "OK" if n >= MIN_IMAGES_PER_CLASS else ("LOW" if n > 0 else "MISSING")
            n_ready += status == "OK"
            print(f"   {it:22s} {n:5d}  {status}")
            rows.append((norm(it), cat, n, status))
            if move_low and status == "LOW":
                os.makedirs(low_dir, exist_ok=True)
                shutil.move(os.path.join(OUT_DIR, norm(it)), os.path.join(low_dir, norm(it)))
        ready += n_ready
        print(f"   -> {n_ready}/{len(items)} items ready")
    csv_path = os.path.join(os.path.dirname(OUT_DIR), "class_to_category.csv")
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["item", "category", "images", "status"])
        w.writerows(rows)
    total = sum(len(v) for v in TAXONOMY.values())
    print(f"\nReady to train: {ready}/{total} items. Add your own photos for the MISSING and LOW items.")
    print("Item -> category list saved to", csv_path)
    return rows

In [ ]:
# Run the fetch. This can take a while the first time (Kaggle, Hugging Face and Open Food Facts downloads).
problems = run_fetch()
print("\nProblems:", problems if problems else "none")

In [ ]:
# Coverage report: which of your 18 categories have enough images?
rows = coverage_report()
DATA_ROOT = OUT_DIR      # the rest of the notebook now trains on this folder

## 3. Find the images and build the splits

This section finds the class folders, lists every image, and creates **train / validation / test** sets. Nothing is shared between the three sets, so the test result is honest.

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def pick(root, names):
    """Return the first sub-folder of `root` whose name is in `names`."""
    for n in names:
        p = os.path.join(root, n)
        if os.path.isdir(p):
            return p
    return None


def find_split_root(base):
    """Walk the dataset folder and return the folder that holds train/test (or the class folders)."""
    for root, dirs, _ in os.walk(base):
        low = [d.lower() for d in dirs]
        if any(n in low for n in ("train", "training")):
            return root, True
    # no train/test folders: look for the first folder that contains class folders full of images
    for root, dirs, files_ in os.walk(base):
        for d in sorted(dirs):
            if any(f.lower().endswith(IMG_EXT) for f in os.listdir(os.path.join(root, d))):
                return root, False
    raise FileNotFoundError(f"No image folders found under {base}")


def list_images(folder, class_names=None):
    """Return class_names, paths, labels for a folder of class sub-folders."""
    if class_names is None:
        class_names = sorted(d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d)))
    rng = random.Random(SEED)
    paths, labels = [], []
    for idx, c in enumerate(class_names):
        cdir = os.path.join(folder, c)
        if not os.path.isdir(cdir):
            continue
        files_ = [f for f in os.listdir(cdir) if f.lower().endswith(IMG_EXT)]
        rng.shuffle(files_)
        if MAX_PER_CLASS:
            files_ = files_[:MAX_PER_CLASS]
        for f in files_:
            paths.append(os.path.join(cdir, f))
            labels.append(idx)
    return class_names, np.array(paths), np.array(labels)


def stratified_split(paths, labels, test_size):
    return train_test_split(paths, labels, test_size=test_size, stratify=labels, random_state=SEED)


root, has_splits = find_split_root(DATA_ROOT)
print("Dataset root:", root)

if has_splits:
    train_dir = pick(root, ["train", "Train", "training", "Training"])
    val_dir = pick(root, ["validation", "Validation", "valid", "val"])
    test_dir = pick(root, ["test", "Test"])
    class_names, p_train, y_train = list_images(train_dir)
    if val_dir:
        _, p_val, y_val = list_images(val_dir, class_names)
    else:
        p_train, p_val, y_train, y_val = stratified_split(p_train, y_train, 0.15)
    if test_dir:
        _, p_test, y_test = list_images(test_dir, class_names)
    else:
        p_val, p_test, y_val, y_test = stratified_split(p_val, y_val, 0.5)
else:
    class_names, p_all, y_all = list_images(root)
    p_train, p_tmp, y_train, y_tmp = stratified_split(p_all, y_all, 0.30)
    p_val, p_test, y_val, y_test = stratified_split(p_tmp, y_tmp, 0.50)

NUM_CLASSES = len(class_names)
print(f"{NUM_CLASSES} classes")
print(f"train: {len(p_train)}  validation: {len(p_val)}  test: {len(p_test)}")
print(class_names)

In [ ]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    return img, label


def make_ds(paths, labels, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_ds(p_train, y_train, training=True)
val_ds = make_ds(p_val, y_val, training=False)
test_ds = make_ds(p_test, y_test, training=False)

## 4. Look at the data

In [ ]:
# Sample images
plt.figure(figsize=(10, 10))
for i, idx in enumerate(np.random.RandomState(SEED).choice(len(p_train), 9, replace=False)):
    img, lab = load_image(p_train[idx], y_train[idx])
    plt.subplot(3, 3, i + 1)
    plt.imshow(img.numpy().astype("uint8"))
    plt.title(class_names[lab])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Images per class in the training set (check that the classes are balanced)
counts = np.bincount(y_train, minlength=NUM_CLASSES)
plt.figure(figsize=(12, 4))
plt.bar(range(NUM_CLASSES), counts)
plt.xticks(range(NUM_CLASSES), class_names, rotation=90)
plt.ylabel("training images")
plt.title("Training images per class")
plt.tight_layout()
plt.show()

## 5. Data augmentation

Augmentation creates slightly different versions of each image during training (flipped, rotated, brighter, zoomed). This helps the model cope with real phone photos. It is only active during training.

In [ ]:
augment = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.15),
        layers.RandomBrightness(0.15, value_range=(0, 255)),
    ],
    name="augmentation",
)

## 6. CNN architecture

### Model A: baseline CNN from scratch
Four convolution blocks. Each block is **Conv2D > BatchNormalization > ReLU > MaxPooling**. The filters grow (32, 64, 128, 256) so early layers learn simple edges and later layers learn shapes and textures. **GlobalAveragePooling** turns the feature maps into one vector, **Dropout** reduces overfitting, and a **softmax** layer gives one probability per food class.

In [ ]:
def build_baseline():
    inputs = keras.Input(shape=IMG_SIZE + (3,), name="image")
    x = augment(inputs)
    x = layers.Rescaling(1.0 / 255)(x)          # pixel values 0-255 -> 0-1
    for filters in (32, 64, 128, 256):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="food_class")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")


baseline = build_baseline()
baseline.summary()

### Model B: transfer learning with MobileNetV2
MobileNetV2 was already trained on millions of images, so it knows general shapes and textures. We keep its layers **frozen**, add our own classification layers, train those first, and then **fine-tune** the top layers of MobileNetV2 with a very small learning rate.

In [ ]:
def build_transfer():
    base = keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,), include_top=False, weights=WEIGHTS
    )
    base.trainable = False                       # frozen for the first training stage

    inputs = keras.Input(shape=IMG_SIZE + (3,), name="image")
    x = augment(inputs)
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)   # MobileNetV2 expects values in [-1, 1]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="food_class")(x)
    return keras.Model(inputs, outputs, name="mobilenetv2_transfer"), base


transfer, base_model = build_transfer()
transfer.summary()

## 7. Train

In [ ]:
def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.SparseTopKCategoricalAccuracy(k=min(3, NUM_CLASSES), name="top3"),
        ],
    )


def make_callbacks(name):
    return [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(f"{name}.keras", monitor="val_accuracy", save_best_only=True),
    ]


def plot_history(histories, title):
    acc, val_acc, loss, val_loss = [], [], [], []
    for h in histories:
        acc += h.history["accuracy"]; val_acc += h.history["val_accuracy"]
        loss += h.history["loss"]; val_loss += h.history["val_loss"]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(acc, label="train"); ax[0].plot(val_acc, label="validation")
    ax[0].set_title(f"{title}: accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(loss, label="train"); ax[1].plot(val_loss, label="validation")
    ax[1].set_title(f"{title}: loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Train Model A (baseline CNN)
compile_model(baseline, 1e-3)
hist_base = baseline.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_BASELINE,
    callbacks=make_callbacks("baseline_cnn"),
)
plot_history([hist_base], "Baseline CNN")

In [ ]:
# Train Model B, stage 1: only the new top layers learn
compile_model(transfer, 1e-3)
hist_t1 = transfer.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_TRANSFER,
    callbacks=make_callbacks("mobilenetv2_stage1"),
)

In [ ]:
# Train Model B, stage 2: fine-tune the last 30 layers of MobileNetV2 with a tiny learning rate
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

compile_model(transfer, 1e-5)
hist_t2 = transfer.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
    callbacks=make_callbacks("mobilenetv2_finetuned"),
)
plot_history([hist_t1, hist_t2], "MobileNetV2 transfer learning")

## 8. Evaluate on the test set

The test set was never used during training, so these numbers show how the model does on new images. We report **top-1 accuracy**, **top-3 accuracy**, **precision / recall / F1** for each class, and a **confusion matrix** (which foods get mixed up).

In [ ]:
def evaluate(model, name):
    probs = model.predict(test_ds, verbose=0)
    y_true = np.concatenate([y.numpy() for _, y in test_ds])
    y_pred = probs.argmax(axis=1)
    top1 = float((y_pred == y_true).mean())
    k = min(3, NUM_CLASSES)
    top3 = float(np.mean([t in np.argsort(p)[-k:] for t, p in zip(y_true, probs)]))
    print(f"{name}: top-1 accuracy = {top1:.3f}   top-3 accuracy = {top3:.3f}")
    return y_true, y_pred, top1, top3


yt_b, yp_b, top1_b, top3_b = evaluate(baseline, "Baseline CNN")
yt_t, yp_t, top1_t, top3_t = evaluate(transfer, "MobileNetV2 (fine-tuned)")

In [ ]:
# Comparison table
print(f"{'Model':28s} {'Top-1':>8s} {'Top-3':>8s}")
print(f"{'Baseline CNN':28s} {top1_b:8.3f} {top3_b:8.3f}")
print(f"{'MobileNetV2 (fine-tuned)':28s} {top1_t:8.3f} {top3_t:8.3f}")

In [ ]:
# Precision, recall and F1 for each class (transfer-learning model)
print(classification_report(yt_t, yp_t, target_names=class_names, zero_division=0))

In [ ]:
# Confusion matrix (transfer-learning model)
cm = confusion_matrix(yt_t, yp_t, labels=range(NUM_CLASSES))
size = min(40, max(6, NUM_CLASSES * 0.4))
fig, ax = plt.subplots(figsize=(size, size))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(class_names, rotation=90)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix")
fig.colorbar(im)
plt.tight_layout(); plt.show()

In [ ]:
# Examples the model got wrong (useful for your failure-case section in the report)
wrong = np.where(yt_t != yp_t)[0][:9]
plt.figure(figsize=(10, 10))
for i, idx in enumerate(wrong):
    img, _ = load_image(p_test[idx], y_test[idx])
    plt.subplot(3, 3, i + 1)
    plt.imshow(img.numpy().astype("uint8"))
    plt.title(f"true: {class_names[yt_t[idx]]}\npred: {class_names[yp_t[idx]]}", fontsize=9)
    plt.axis("off")
plt.tight_layout(); plt.show()

## 9. Try it on your own photo

Take a photo of a fruit or vegetable on your phone, upload it, and see the top 3 guesses. **Real phone photos are the most important test** of your project.

In [ ]:
# [COLAB] Upload a photo and predict
from google.colab import files
up = files.upload()
for fn in up:
    img = keras.utils.load_img(fn, target_size=IMG_SIZE)
    x = keras.utils.img_to_array(img)[None, ...]
    probs = transfer.predict(x, verbose=0)[0]
    top = np.argsort(probs)[::-1][:3]
    plt.imshow(img); plt.axis("off"); plt.show()
    for i in top:
        print(f"{class_names[i]:20s} {probs[i] * 100:5.1f}%")

## 10. Save the model

The saved model and the class list are what your web app (Flask or FastAPI) will load later.

In [ ]:
transfer.save("food_cnn_mobilenetv2.keras")
with open("class_names.json", "w") as f:
    json.dump(class_names, f)
print("Saved: food_cnn_mobilenetv2.keras and class_names.json")

In [ ]:
# [COLAB] Copy the files to Google Drive so you keep them after the session ends
from google.colab import drive
import shutil
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/food_cnn_outputs", exist_ok=True)
for fn in ["food_cnn_mobilenetv2.keras", "class_names.json", "baseline_cnn.keras"]:
    if os.path.exists(fn):
        shutil.copy(fn, "/content/drive/MyDrive/food_cnn_outputs/")
print("Copied to Drive: food_cnn_outputs")

In [ ]:
# Optional: convert to TensorFlow Lite for a future mobile app
converter = tf.lite.TFLiteConverter.from_keras_model(transfer)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open("food_cnn.tflite", "wb") as f:
    f.write(tflite_model)
print("TFLite size (MB):", round(len(tflite_model) / 1e6, 2))

## Next steps

1. Add your own phone photos (100 to 200 per food, especially Indian items) to the dataset and train again.
2. Keep a test set made only of your own phone photos and report its accuracy separately.
3. Record the results (top-1, top-3, F1, confusion matrix, failure cases) in your evaluation report.
4. Link each class name to a nutrition entry (USDA FoodData Central or the Indian Food Composition Tables).